# 抗体序列读取与特征提取

本 Notebook 演示：
- 从 FASTA 文件读取抗体序列（Biopython SeqIO）
- 用 ProtParam 计算分子量、等电点、疏水性、氨基酸组成
- 用 Seaborn 热图可视化特征矩阵

In [ ]:
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
# Day 1
def load_sequences(fasta_path):
    sequences = []
    for record in SeqIO.parse(fasta_path, "fasta"):
        sequences.append({
            "id": record.id,
            "seq": str(record.seq)
        })
    return sequences

# 读取数据
seqs = load_sequences("../data/你的文件名.fasta")
print(f"读取 {len(seqs)} 条序列")
print(f"第一条: {seqs[0]['id']}, 长度: {len(seqs[0]['seq'])}")

In [ ]:
# Day 4
def extract_features(sequences):
    feature_list = []
    for item in sequences:
        seq_str = item["seq"]
        analysed = ProteinAnalysis(seq_str)
        features = {
            "ID": item["id"],
            "Length": len(seq_str),
            "Molecular_Weight": analysed.molecular_weight(),
            "Isoelectric_Point": analysed.isoelectric_point(),
            "GRAVY": analysed.gravy(),
            "Aromaticity": analysed.aromaticity(),
            "Instability_Index": analysed.instability_index(),
        }
        aa_comp = analysed.amino_acids_percent
        features.update(aa_comp)
        feature_list.append(features)
    return pd.DataFrame(feature_list)

df_features = extract_features(seqs)
print(f"特征矩阵: {df_features.shape[0]} 行 × {df_features.shape[1]} 列")
df_features.head()

In [ ]:
numeric_cols = df_features.select_dtypes(include='number').columns.tolist()
X = df_features[numeric_cols]

plt.figure(figsize=(14, 6))
sns.heatmap(X, annot=True, fmt=".2f", cmap="RdYlBu_r",
            xticklabels=X.columns, yticklabels=df_features["ID"])
plt.title("抗体序列特征热图")
plt.tight_layout()
plt.show()

## 小结
成功将抗体序列转化为数值特征矩阵。热图直观展示不同序列在分子量、等电点、疏水性等维度的差异。